# LLMMultiRoundRouter - Inference

This notebook demonstrates how to use the **LLMMultiRoundRouter** for multi-round query processing.

## Overview

LLMMultiRoundRouter uses LLM prompts for both query decomposition and routing decisions.
Unlike KNN-based routers, it doesn't require training - it uses LLM reasoning directly.

**Pipeline**:
1. **Decompose + Route**: LLM breaks query into sub-queries AND routes each in one step
2. **Execute**: Call routed model APIs for each sub-query
3. **Aggregate**: LLM combines responses into final answer

**Key Features**:
- No training required (LLM-based reasoning)
- Single-step decomposition and routing
- Model descriptions guide routing decisions
- Supports both local vLLM and API-based inference

## 1. Environment Setup

In [1]:
# For Google Colab
import os

!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter
!pip install -e .
!pip install pyyaml openai transformers torch


Cloning into 'LLMRouter'...
remote: Enumerating objects: 6017, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 6017 (delta 105), reused 96 (delta 90), pack-reused 5837 (from 2)
Receiving objects: 100% (6017/6017), 89.41 MiB | 55.09 MiB/s, done.
Resolving deltas: 100% (2946/2946), done.
Updating files: 100% (288/288), done.
/home/zhongjie/LLMRouter
Obtaining file:///home/zhongjie/LLMRouter
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llmrouter-lib (pyproject.toml) ... done
  Created wheel for llmrouter-lib: filename=llmrouter_lib-0.2.0-0.editable-py3-none-any.whl size=15709 sha256=ffb5b5840bbaf72b8620517fdb46a164069b08683d5a38bae987263abad5747a
  Stored in directory: /tmp/pip-ephem-wheel-cache-f28j99y7/wheels/82/4a/fd/59c4aec93c356c380

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [3]:
from llmrouter.utils import setup_environment
import yaml

setup_environment()
print("Environment setup complete!")

Environment setup complete!


## 2. Configuration

LLMMultiRoundRouter requires:

| Parameter | Description | Required |
|-----------|-------------|----------|
| `llm_data` | LLM candidates with descriptions | Yes |
| `base_model` | LLM for decomposition/aggregation | Yes |
| `api_endpoint` | API endpoint for execution | Yes |
| `use_local_llm` | Use vLLM for local inference | No |

In [6]:
# Create configuration for LLMMultiRoundRouter
llm_multi_config = {
    "data_path": {
        "query_data_test": "data/example_data/query_data/default_query_test.jsonl",
        "routing_data_test": "data/example_data/routing_data/default_routing_test_data.jsonl",
        "llm_data": "data/example_data/llm_candidates/default_llm.json"
    },
    "base_model": "qwen/qwen2.5-7b-instruct",
    "use_local_llm": False,
    "api_endpoint": os.environ.get("API_ENDPOINT", "https://api.openai.com/v1")
}

# Save config
CONFIG_PATH = "configs/model_config_train/llmmultiroundrouter_temp.yaml"
os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(llm_multi_config, f, default_flow_style=False)

print("Configuration:")
print("=" * 50)
print(yaml.dump(llm_multi_config, default_flow_style=False))

Configuration:
api_endpoint: https://integrate.api.nvidia.com/v1
base_model: qwen/qwen2.5-7b-instruct
data_path:
  llm_data: data/example_data/llm_candidates/default_llm.json
  query_data_test: data/example_data/query_data/default_query_test.jsonl
  routing_data_test: data/example_data/routing_data/default_routing_test_data.jsonl
use_local_llm: false



## 3. Initialize Router

In [7]:
from llmrouter.models.llmmultiroundrouter import LLMMultiRoundRouter

try:
    router = LLMMultiRoundRouter(yaml_path=CONFIG_PATH)
    print("Router initialized successfully!")
    print(f"Base model: {router.base_model}")
    print(f"Use local LLM: {router.use_local_llm}")
    print(f"Number of LLM candidates: {len(router.llm_data)}")
except Exception as e:
    print(f"Error initializing router: {e}")

✅ MetaRouter initialized successfully (YAML + data loaded).
Router initialized successfully!
Base model: qwen/qwen2.5-7b-instruct
Use local LLM: False
Number of LLM candidates: 7


In [8]:
# Display LLM candidates with their descriptions
print("Available LLM Candidates:")
print("=" * 60)

for name, info in router.llm_data.items():
    print(f"\n{name}:")
    if 'description' in info:
        print(f"  Description: {info['description'][:100]}...")
    if 'size' in info:
        print(f"  Size: {info['size']}B parameters")
    if 'capabilities' in info:
        print(f"  Capabilities: {info['capabilities']}")

Available LLM Candidates:

qwen2.5-7b-instruct:
  Size: 7BB parameters

llama-3.1-8b-instruct:
  Size: 8BB parameters

mistral-7b-instruct-v0.3:
  Size: 7BB parameters

llama-3.3-nemotron-super-49b-v1:
  Size: 49BB parameters

llama3-70b-instruct:
  Size: 70BB parameters

mixtral-8x7b-instruct-v0.1:
  Size: 45BB parameters

mixtral-8x22b-instruct-v0.1:
  Size: 141BB parameters


## 4. Chat Mode (Simple Queries)

For simple queries, pass a string and get a string response.

In [9]:
# Chat mode - simple string input/output
query = "What are the main causes of climate change and what solutions exist?"

print(f"Query: {query}")
print("=" * 60)

try:
    response = router.route_single(query)
    print(f"\nResponse:\n{response}")
except Exception as e:
    print(f"Error: {e}")
    print("\nNote: LLMMultiRoundRouter requires API access for LLM calls.")

Query: What are the main causes of climate change and what solutions exist?

Response:
The main causes of climate change are primarily human activities, which include:

1. **Burning Fossil Fuels**: The combustion of coal, oil, and gas for energy releases large amounts of carbon dioxide (CO₂) and other greenhouse gases into the atmosphere.
2. **Deforestation**: Cutting down forests reduces the number of trees that can absorb CO₂, and often leads to the release of stored carbon when trees are burned or decompose.
3. **Agriculture**: Practices such as livestock farming and rice cultivation release methane, a potent greenhouse gas. The use of fertilizers also leads to increased nitrous oxide emissions.
4. **Industrial Processes**: Various industrial activities release greenhouse gases, including CO₂, methane, and fluorinated gases.
5. **Waste Management**: Landfills produce methane as organic waste decomposes.

To address these causes, solutions can be broadly categorized into mitigation a

## 5. Evaluation Mode (With Metrics)

For evaluation, pass a dict with task_name and ground_truth.

In [10]:
# Evaluation mode - dict input with metrics
eval_query = {
    "query": "What is the largest planet in our solar system?",
    "task_name": "trivia",
    "ground_truth": "Jupiter"
}

print(f"Query: {eval_query['query']}")
print(f"Task: {eval_query['task_name']}")
print(f"Ground Truth: {eval_query['ground_truth']}")
print("=" * 60)

try:
    result = router.route_single(eval_query)
    
    print(f"\nResponse: {result.get('response', 'N/A')}")
    print(f"Success: {result.get('success', False)}")
    print(f"Prompt Tokens: {result.get('prompt_tokens', 0)}")
    print(f"Completion Tokens: {result.get('completion_tokens', 0)}")
    if 'task_performance' in result:
        print(f"Task Performance: {result['task_performance']:.2f}")
except Exception as e:
    print(f"Error: {e}")

Query: What is the largest planet in our solar system?
Task: trivia
Ground Truth: Jupiter

Response: To determine the largest planet in our solar system, let's analyze the provided information step by step:

1. **Sub-query: What is the largest planet in our solar system?**
   - Response: The largest planet in our solar system is Jupiter.

2. **Sub-query: What is the name of the largest planet in our solar system?**
   - Response: The largest planet in our solar system is Jupiter.

3. **Sub-query: What is the size of the largest planet in our solar system?**
   - Response: The largest planet in our solar system is Jupiter, with a diameter of approximately 86,881 miles (139,822 kilometers).

From these sub-queries, we can clearly see that Jupiter is consistently identified as the largest planet in our solar system. The additional information about its size further confirms this.

Therefore, the largest planet in our solar system is Jupiter.

<answer>Jupiter</answer>
Success: True
Prompt 

## 6. Batch Processing

In [11]:
# Batch processing
batch_queries = [
    {"query": "Explain quantum computing."},
    {"query": "What is the difference between AI and ML?"},
    {"query": "How does blockchain technology work?"},
]

print(f"Processing {len(batch_queries)} queries...")
print("=" * 60)


try:
    results = router.route_batch(batch_queries)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. Query: {result.get('query', 'N/A')[:50]}...")
        print(f"   Success: {result.get('success', False)}")
        response = result.get('response', 'N/A')
        print(f"   Response: {response[:100] if response else 'N/A'}...")
except Exception as e:
    print(f"Error: {e}")

Processing 3 queries...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


1. Query: Explain quantum computing....
   Success: True
   Response: API Error: litellm.Timeout: APITimeoutError - Request timed out. Error_str: Request timed out....

2. Query: What is the difference between AI and ML?...
   Success: True
   Response: To understand the difference between AI and ML, let's break it down step by step:

1. **Scope**:
   ...

3. Query: How does blockchain technology work?...
   Success: True
   Response: Blockchain technology works through a combination of decentralized networks, cryptographic technique...


## 7. Task-Specific Routing

LLMMultiRoundRouter supports task-specific prompts for evaluation.

In [13]:
# Multiple choice task
mc_query = {
    "query": "What is the capital of Australia? A) Sydney B) Melbourne C) Canberra D) Perth",
    "task_name": "commonsense_qa",
    "choices": {
        "label": ["A", "B", "C", "D"],
        "text": ["Sydney", "Melbourne", "Canberra", "Perth"]
    },
    "ground_truth": "C"
}

print(f"Multiple Choice Query:")
print(f"Question: {mc_query['query']}")
print(f"Ground Truth: {mc_query['ground_truth']}")
print("=" * 60)

try:
    results = router.route_batch([mc_query], task_name="commonsense_qa")
    result = results[0]
    
    print(f"\nResponse: {result.get('response', 'N/A')}")
    if 'task_performance' in result:
        print(f"Correct: {result['task_performance'] == 1.0}")
except Exception as e:
    print(f"Error: {e}")

Multiple Choice Query:
Question: What is the capital of Australia? A) Sydney B) Melbourne C) Canberra D) Perth
Ground Truth: C

Response: <answer>C</answer>


## 8. Understanding the Pipeline

Let's examine how LLMMultiRoundRouter works.

In [ ]:
print("LLMMultiRoundRouter Pipeline:")
print("=" * 60)

print("""
┌─────────────────────────────────────────────────────────────┐
│                    Input Query                              │
│  "What are the causes of climate change and solutions?"    │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│              Step 1: Decompose + Route                      │
│  LLM breaks query into sub-queries AND routes each          │
│                                                             │
│  Output format: <sub-query>: <model-name>                   │
│  • "What causes climate change?": Qwen-7B                   │
│  • "What solutions exist for climate change?": Llama-70B    │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│              Step 2: Execute Sub-queries                    │
│  Call routed model API for each sub-query                   │
│                                                             │
│  • Qwen-7B → "Greenhouse gases, deforestation..."           │
│  • Llama-70B → "Renewable energy, carbon capture..."        │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│              Step 3: Aggregate Responses                    │
│  LLM combines sub-responses into coherent final answer      │
│                                                             │
│  "Climate change is caused by greenhouse gases...           │
│   Solutions include renewable energy and..."                │
└─────────────────────────────────────────────────────────────┘
""")

print("\nKey Advantages:")
print("• No training required - uses LLM reasoning directly")
print("• Model descriptions guide intelligent routing decisions")
print("• Single-step decomposition and routing for efficiency")
print("• Flexible aggregation based on task type")

## 9. Comparison with KNNMultiRoundRouter

In [ ]:
print("LLMMultiRoundRouter vs KNNMultiRoundRouter:")
print("=" * 60)

comparison = """
| Feature              | LLMMultiRoundRouter     | KNNMultiRoundRouter     |
|----------------------|-------------------------|-------------------------|
| Training Required    | No                      | Yes (KNN fitting)       |
| Routing Method       | LLM reasoning           | KNN on embeddings       |
| Model Selection      | Based on descriptions   | Based on similarity     |
| Decomposition        | Same LLM call           | Separate LLM call       |
| Flexibility          | High (prompt-based)     | Medium (learned)        |
| Inference Cost       | Higher (more LLM calls) | Lower (KNN is fast)     |
| Cold Start           | Works immediately       | Needs training data     |
"""
print(comparison)

## 10. File-Based Inference

Load queries from a file and save results.

In [14]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/llmmultiroundrouter_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries (limit to 5 for demo due to API costs)
    try:
        file_results = router.route_batch(file_queries[:5])
        print(f"Routed {len(file_results)} queries")
        
        # Save results
        save_results_to_file(file_results, OUTPUT_FILE)
        
        # Show sample results
        print(f"\nSample results:")
        for i, result in enumerate(file_results[:3], 1):
            print(f"  {i}. {result.get('query', '')[:40]}...")
            print(f"     Success: {result.get('success', False)}")
    except Exception as e:
        print(f"Error during batch routing: {e}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

Loaded 706 queries from: data/example_data/query_data/default_query_test.jsonl

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Routed 5 queries
Results saved to: outputs/llmmultiroundrouter_results.jsonl

Sample results:
  1. Q: There are 4 houses in a row, numbered...
     Success: True
  2. Q: There are 3 houses in a row, numbered...
     Success: True
  3. Q: There are 3 houses in a row, numbered...
     Success: True


## Summary

**LLMMultiRoundRouter** provides:
- Zero-shot multi-round routing (no training)
- LLM-based decomposition and routing in one step
- Model description-guided routing decisions
- Flexible aggregation for different task types

**Use Cases**:
- Quick prototyping without training data
- Complex queries requiring expert routing decisions
- When model descriptions are more reliable than embeddings
- Low-volume, high-quality routing needs

**Requirements**:
- API access for LLM calls
- Optional: vLLM for local inference
- Model descriptions in llm_data (recommended)